# 07 · Spatial-window LIANA rank aggregation

This notebook estimates ligand–receptor activity in overlapping adaptive spatial windows, collapses those windows to one edge vector per biological sample, and compares conditions at the sample level.

Configure the cell scope and ligand–receptor resource, audit the input, and run a small smoke test before launching the full resumable analysis.

In [ ]:
import json
import os
import shlex
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import liana as li
from IPython.display import Image, Markdown, display

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
SOURCE_ROOT = str(REPO_ROOT / "src")
if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)

from spatial_workflow.config import load_config, resolve_path
from spatial_workflow.liana_edge_features import top_edge_features
from spatial_workflow.liana_window import (
    WindowParameters,
    analyze_window_rankagg,
    audit_liana_input,
    liana_window_status,
    load_cellchat_resource,
    load_rankagg_resource,
    load_window_analysis,
    parameter_table,
    prepare_chord_input,
    render_chord,
    run_window_rankagg,
)
from spatial_workflow.liana_window_plotting import (
    plot_pathway_driver_summary,
    plot_pathway_ranking,
)

CONFIG_PATH = REPO_ROOT / "configs" / "local.yaml"
config = load_config(CONFIG_PATH)
schema = config["schema"]
results_root = resolve_path(CONFIG_PATH, config["paths"]["results_root"])
cellcharter = config["cellcharter"]
INPUT_H5AD = resolve_path(
    CONFIG_PATH,
    Path(cellcharter["output_dir"]) / f"{cellcharter['output_name']}.h5ad",
    root=results_root,
)
OUTPUT_DIR = results_root / "07_liana_window_rankagg"
ANALYSIS_DIR = OUTPUT_DIR / "analysis"
CELLCHAT_CSV = Path(
    os.environ.get(
        "CELLCHAT_CSV",
        REPO_ROOT / "resources" / "CellChatDB_interaction.csv",
    )
)
CHORD_SCRIPT = REPO_ROOT / "scripts" / "plot_liana_window_chord.R"

print("input:", INPUT_H5AD)
print("output:", OUTPUT_DIR)
print("CellChat:", CELLCHAT_CSV)

## Analysis scope and parameters

`COMPARTMENTS=None` is a true whole-sample analysis: all cells in each biological sample are eligible and window construction does not use compartment labels. An explicit compartment list filters cells first and pools the retained domains; it does not run one LIANA analysis per domain. `CELL_TYPES` similarly filters cells before windows are constructed.

Choose a CellChat, custom, or native LIANA resource and optionally restrict pathways, annotations, or ligand–receptor pairs. For CellChat mode, set the `CELLCHAT_CSV` environment variable or edit `CELLCHAT_CSV` in the setup cell.

`WindowParameters` controls window size, expression support, spatial weighting, and LIANA permutations. Review the printed resource and parameter audits before running the analysis.

In [ ]:
# None means all available values. These filters are applied before windows.
CELL_TYPES = None          # e.g. ["Astrocyte", "Microglia"]
COMPARTMENTS = None        # true whole sample; or e.g. ["0", "3"] pooled

# LR_RESOURCE_MODE: "cellchat", "custom", or "liana".
LR_RESOURCE_MODE = "cellchat"
CUSTOM_LR_CSV = None        # CSV with ligand/receptor columns
LIANA_RESOURCE_NAME = "mouseconsensus"
LR_PATHWAYS = None          # pathway_name values; CellChat/annotated custom only
LR_ANNOTATIONS = None       # annotation values; CellChat/annotated custom only
LR_PAIRS = None             # exact ligand^receptor strings
INCLUDE_NON_PROTEIN = False # CellChat v2 default; True adds metabolic/synaptic rows
# Pathway annotation used downstream; for an annotated custom LR CSV,
# set this to CUSTOM_LR_CSV. Native mouseconsensus has no pathways, so
# the CellChat default annotates only its overlapping pairs.
PATHWAY_METADATA_CSV = CELLCHAT_CSV


PARAMETERS = WindowParameters(
    adaptive_k=60,
    grid_stride=100.0,
    expr_prop=0.1,
    min_cells=5,
    min_required_groups=2,
    spatial_bandwidth=250.0,
    spatial_kernel="gaussian",
    spatial_trim_fraction=0.1,
    n_perms=200,
    seed=1337,
    n_jobs=8,
    anchor_groups=(),
    anchor_k_target=60,
    anchor_min_cells=2,
    anchor_max_radius=150.0,
    anchor_dedup_distance=60.0,
    return_all_lrs=False,
)
display(parameter_table(PARAMETERS))

resource, resource_metadata = load_rankagg_resource(
    li,
    mode=LR_RESOURCE_MODE,
    cellchat_csv=CELLCHAT_CSV,
    custom_lr_csv=CUSTOM_LR_CSV,
    liana_resource_name=LIANA_RESOURCE_NAME,
    pathways=LR_PATHWAYS,
    annotations=LR_ANNOTATIONS,
    lr_pairs=LR_PAIRS,
    include_non_protein=INCLUDE_NON_PROTEIN,
)
display(
    pd.Series(
        {
            "selected resource metadata rows": len(resource_metadata),
            "distinct LIANA gene-complex tests": len(resource),
            "annotated pathways": resource_metadata["pathway_name"].nunique(),
            "annotations": resource_metadata["annotation"].nunique(),
        },
        name="resource audit",
    ).to_frame()
)

## Input audit

This read-only check reports the samples, conditions, selected cells, and eligible annotations after applying the requested scope. Use it to catch missing or unexpectedly sparse groups before launching LIANA.

In [ ]:
RUN_INPUT_AUDIT = True
if RUN_INPUT_AUDIT:
    input_audit = audit_liana_input(
        INPUT_H5AD,
        sample_key=schema["sample_key"],
        condition_key=schema["condition_key"],
        group_key=schema["cell_type_key"],
        spatial_key=schema["spatial_key"],
        compartment_key=schema.get("spatial_domain_key"),
        cell_types=CELL_TYPES,
        compartments=COMPARTMENTS,
    )
    display(input_audit["shape"].to_frame("value"))
    display(input_audit["samples"])
    display(input_audit["cell_groups"].head(50))

## Run or resume sample-level LIANA

This is the expensive stage. Outputs and manifests are written separately for each biological sample, allowing completed samples to be reused.

Use a small sample list or window limit only for a clean smoke-test directory. After reviewing the command preview, enable the run switch to create the resumable tmux job.

In [ ]:
RUN_LIANA = False
LAUNCH_TMUX = False         # set True to detach the expensive run
TMUX_SESSION = "liana_window_rankagg"
TMUX_LOG = OUTPUT_DIR / "logs" / f"{TMUX_SESSION}.log"
SAMPLES = None             # e.g. ["vap_46_igg"] for a smoke test
LIMIT_WINDOWS = None       # e.g. 2 for smoke testing only
OVERWRITE_LIANA = False

def _append_repeated(arguments, flag, values):
    for value in values or []:
        arguments.extend([flag, str(value)])

cli_arguments = [
    sys.executable,
    str(REPO_ROOT / "scripts" / "run_liana_window_rankagg.py"),
    "--config",
    str(CONFIG_PATH),
    "run",
    "--input-h5ad",
    str(INPUT_H5AD),
    "--output-dir",
    str(OUTPUT_DIR),
    "--resource-mode",
    LR_RESOURCE_MODE,
    "--liana-resource-name",
    LIANA_RESOURCE_NAME,
    "--adaptive-k",
    str(PARAMETERS.adaptive_k),
    "--grid-stride",
    str(PARAMETERS.grid_stride),
    "--expr-prop",
    str(PARAMETERS.expr_prop),
    "--min-cells",
    str(PARAMETERS.min_cells),
    "--min-required-groups",
    str(PARAMETERS.min_required_groups),
    "--spatial-bandwidth",
    str(PARAMETERS.spatial_bandwidth),
    "--spatial-kernel",
    PARAMETERS.spatial_kernel,
    "--spatial-trim-fraction",
    str(PARAMETERS.spatial_trim_fraction),
    "--n-perms",
    str(PARAMETERS.n_perms),
    "--seed",
    str(PARAMETERS.seed),
    "--n-jobs",
    str(PARAMETERS.n_jobs),
    "--anchor-k-target",
    str(PARAMETERS.anchor_k_target),
    "--anchor-min-cells",
    str(PARAMETERS.anchor_min_cells),
    "--anchor-max-radius",
    str(PARAMETERS.anchor_max_radius),
    "--anchor-dedup-distance",
    str(PARAMETERS.anchor_dedup_distance),
]
if LR_RESOURCE_MODE == "cellchat":
    cli_arguments.extend(["--cellchat-csv", str(CELLCHAT_CSV)])
elif LR_RESOURCE_MODE == "custom":
    if CUSTOM_LR_CSV is None:
        raise ValueError("CUSTOM_LR_CSV is required for custom LR mode")
    cli_arguments.extend(["--custom-lr-csv", str(CUSTOM_LR_CSV)])
_append_repeated(cli_arguments, "--sample", SAMPLES)
_append_repeated(cli_arguments, "--cell-type", CELL_TYPES)
_append_repeated(cli_arguments, "--compartment", COMPARTMENTS)
_append_repeated(cli_arguments, "--pathway", LR_PATHWAYS)
_append_repeated(cli_arguments, "--annotation", LR_ANNOTATIONS)
_append_repeated(cli_arguments, "--lr-pair", LR_PAIRS)
_append_repeated(cli_arguments, "--anchor-group", PARAMETERS.anchor_groups)
if INCLUDE_NON_PROTEIN:
    cli_arguments.append("--include-non-protein")
if PARAMETERS.return_all_lrs:
    cli_arguments.append("--return-all-lrs")
if OVERWRITE_LIANA:
    cli_arguments.append("--overwrite")
if LIMIT_WINDOWS is not None:
    cli_arguments.extend(["--limit-windows-per-sample", str(LIMIT_WINDOWS)])

print("CLI preview:", shlex.join(cli_arguments))
if RUN_LIANA and LAUNCH_TMUX:
    TMUX_LOG.parent.mkdir(parents=True, exist_ok=True)
    session_exists = subprocess.run(
        ["tmux", "has-session", "-t", TMUX_SESSION],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=False,
    ).returncode == 0
    if session_exists:
        raise RuntimeError(f"tmux session already exists: {TMUX_SESSION}")
    shell_command = (
        f"cd {shlex.quote(str(REPO_ROOT))} && "
        f"{shlex.join(cli_arguments)} 2>&1 | tee {shlex.quote(str(TMUX_LOG))}"
    )
    subprocess.run(
        ["tmux", "new-session", "-d", "-s", TMUX_SESSION, "bash", "-lc", shell_command],
        check=True,
    )
    display(pd.Series({
        "tmux_session": TMUX_SESSION,
        "log": str(TMUX_LOG),
        "attach": f"tmux attach -t {TMUX_SESSION}",
    }))
elif RUN_LIANA:
    run_status = run_window_rankagg(
        INPUT_H5AD,
        CELLCHAT_CSV,
        OUTPUT_DIR,
        sample_key=schema["sample_key"],
        condition_key=schema["condition_key"],
        group_key=schema["cell_type_key"],
        spatial_key=schema["spatial_key"],
        compartment_key=schema.get("spatial_domain_key"),
        cell_types=CELL_TYPES,
        compartments=COMPARTMENTS,
        pathways=LR_PATHWAYS,
        annotations=LR_ANNOTATIONS,
        lr_pairs=LR_PAIRS,
        resource_mode=LR_RESOURCE_MODE,
        custom_lr_csv=CUSTOM_LR_CSV,
        liana_resource_name=LIANA_RESOURCE_NAME,
        include_non_protein=INCLUDE_NON_PROTEIN,
        parameters=PARAMETERS,
        samples=SAMPLES,
        overwrite=OVERWRITE_LIANA,
        limit_windows_per_sample=LIMIT_WINDOWS,
    )
    display(run_status)
else:
    print("Set RUN_LIANA=True; use LAUNCH_TMUX=True for a detached run.")

artifact_status = liana_window_status(OUTPUT_DIR)
display(artifact_status if not artifact_status.empty else pd.DataFrame(
    {"status": ["No completed sample manifests yet"]}
))

## Build the tested edge universe

The analysis first defines supported source–target–ligand–receptor edges across samples, then completes missing edge–sample combinations with the configured inactive value before calculating condition statistics.

The displayed audit records support thresholds, zero filling, contrasts, and the exact permutation resolution so each tested edge can be interpreted reproducibly.

In [ ]:
CONDITIONS = ["naive_igg", "vap_igg", "vap_ab"]
MIN_CONDITION_SAMPLES = 3
RANK_COLUMN = "magnitude_rank"
ZERO_FILL_MISSING = True   # default; False uses observed/complete cases
RUN_PAIRWISE_ANALYSIS = False
OVERWRITE_ANALYSIS = False

if RUN_PAIRWISE_ANALYSIS:
    analysis_tables = analyze_window_rankagg(
        OUTPUT_DIR,
        PATHWAY_METADATA_CSV,
        ANALYSIS_DIR,
        conditions=CONDITIONS,
        min_condition_samples=MIN_CONDITION_SAMPLES,
        rank_column=RANK_COLUMN,
        zero_fill_missing=ZERO_FILL_MISSING,
        include_non_protein=INCLUDE_NON_PROTEIN,
        overwrite=OVERWRITE_ANALYSIS,
    )
    display(
        pd.Series(
            {name: table.shape for name, table in analysis_tables.items()},
            name="shape",
        ).to_frame()
    )
else:
    print("Set RUN_PAIRWISE_ANALYSIS=True after all intended samples finish.")

## Review edge-level changes

Each row is one source–target–ligand–receptor edge for one contrast. `mean_activity_a` and `mean_activity_b` average its sample-level activity within each condition, and `difference_b_minus_a` is the signed effect: positive values indicate higher activity in condition B.

- `exact_permutation_p` is a two-sided exhaustive reassignment of biological-sample condition labels for that mean difference. It does not treat windows or cells as replicates. The value is raw; `exact_permutation_q` is the Benjamini–Hochberg adjustment across tested edges in the contrast.
- `welch_p` is the two-sided Welch t-test on the same sample-level activities and allows unequal condition variances; `welch_q` is its Benjamini–Hochberg adjustment. With few samples, use the exact test as the primary small-sample result and Welch as a complementary parametric check.
- `presence_class` and the observed-sample counts describe whether LIANA emitted the edge with adequate support in both conditions or whether it appeared, disappeared, or was sparse. They describe support, not statistical significance.

With four samples per condition, the minimum attainable two-sided exact p-value is `0.02857`. Interpret p-values together with effect direction, magnitude, support, and the missing-edge policy.

In [ ]:
analysis_tables = load_window_analysis(ANALYSIS_DIR) if (
    ANALYSIS_DIR / "manifest.json"
).exists() else None

if analysis_tables is not None:
    edge_pairwise = analysis_tables["edge_pairwise"]
    selected_contrast = edge_pairwise["contrast"].iloc[0]
    edge_review = edge_pairwise[
        edge_pairwise["contrast"].eq(selected_contrast)
        & edge_pairwise["tested"]
    ].sort_values(
        ["exact_permutation_p", "abs_difference"],
        ascending=[True, False],
    )
    display(edge_review.head(50))
    display(
        edge_review.groupby("presence_class", observed=True)
        .agg(
            n_edges=("edge_id", "nunique"),
            median_abs_difference=("abs_difference", "median"),
            raw_p_lt_0_05=("exact_permutation_p", lambda x: (x < 0.05).sum()),
            fdr_q_lt_0_05=("exact_permutation_q", lambda x: (x < 0.05).sum()),
        )
        .sort_values("n_edges", ascending=False)
    )
else:
    print("Analysis tables are not present yet.")

## Summarize significant edge features

For a selected contrast and exact-p threshold, this section counts significant ligands, receptors, ligand–receptor pairs, pathways, sources, targets, and source–target pairs. Counts refer to unique tested edges.

In [ ]:
EDGE_FEATURE_CONTRASTS = (
    "vap_igg_vs_naive_igg",
    "vap_ab_vs_vap_igg",
)
EDGE_FEATURE_THRESHOLD = 0.05
EDGE_FEATURE_TOP_N = 20

edge_feature_tables_by_contrast = None
if analysis_tables is not None:
    edge_feature_tables_by_contrast = {}
    for contrast in EDGE_FEATURE_CONTRASTS:
        display(Markdown(f"### {contrast}"))
        feature_tables = top_edge_features(
            analysis_tables["edge_pairwise"],
            contrast=contrast,
            threshold=EDGE_FEATURE_THRESHOLD,
            top_n=EDGE_FEATURE_TOP_N,
        )
        edge_feature_tables_by_contrast[contrast] = feature_tables
        for table_name, table in feature_tables.items():
            display(Markdown(f"#### {table_name.replace("_", " ").title()}"))
            display(table)
else:
    print("Analysis tables are not present yet.")

## Rank pathway changes

Pathways are ranked by `rms_centroid_distance` after requiring either pathway-level exact p-value to pass the selected threshold.

- `mean_activity_exact_p` first averages all eligible pathway edges within each sample, then exhaustively reassigns the biological-sample condition labels and tests `mean_activity_difference_b_minus_a`, the difference between condition means. It is most sensitive to a coherent overall increase or decrease across the pathway.
- `rms_centroid_distance` is the root-mean-square difference between the two condition mean activity vectors across the pathway's edges. It is an unsigned effect size, so large increases and decreases cannot cancel each other.
- `rms_exact_p` exhaustively reassigns the sample labels and asks how often the resulting RMS centroid distance is at least as large as the observed distance. It detects a change in the pathway's multiedge signaling profile even when the average pathway activity changes little.

Both exact p-values use biological samples as replicates and are raw; `mean_activity_exact_q` and `rms_exact_q` are their Benjamini–Hochberg adjustments across pathways in the contrast. Use the bar chart to select a pathway, then inspect its contributing ligand–receptor identities and directed cell-pair edges below.

In [ ]:
# Ranking options: rms_centroid_distance, rms_exact_p, mean_activity_exact_p,
# or mean_activity_difference_b_minus_a.
PATHWAY_RANK_BY = "rms_centroid_distance"
PATHWAY_TOP_N = 20
PATHWAY_EXACT_P_MAX = 0.05  # mean-activity OR RMS exact p must be below this

if analysis_tables is not None:
    pathway_pairwise = analysis_tables["pathway_pairwise"]
    pathway_rank_figure, pathway_rank_axis, pathway_review = plot_pathway_ranking(
        pathway_pairwise,
        contrast=selected_contrast,
        rank_by=PATHWAY_RANK_BY,
        top_n=PATHWAY_TOP_N,
        exact_p_max=PATHWAY_EXACT_P_MAX,
    )
    selected_pathway = pathway_review.iloc[0]["pathway_name"]
    display(pathway_rank_figure)
    plt.close(pathway_rank_figure)

    pathway_columns = [
        "display_rank", "pathway_name", "mean_activity_difference_b_minus_a",
        "mean_activity_exact_p", "rms_centroid_distance", "rms_exact_p",
    ]
    display(pathway_review[pathway_columns])
else:
    print("Analysis tables are not present yet.")

## Inspect one pathway

The driver bars group the pathway's directed cell-pair edges by ligand–receptor identity. **Share of absolute effect** is `100 × sum(abs(edge difference))` for that ligand–receptor pair divided by the same sum over every edge in the pathway. It therefore measures how much of the pathway's total change magnitude is carried by that pair—not its statistical significance or direction. Shares across all pathway drivers sum to 100%; a top-N display may sum to less.

`signed_effect_sum` gives the net direction, while `direction_balance` divides the signed sum by the absolute sum: `+1` means all contributing edges increase in condition B, `-1` means all decrease, and values near zero indicate cancellation. The adjacent bubble plot shows the individual source-to-target edges, with color encoding signed activity change and size encoding edge-level significance.

Change the selected pathway and contrast in the control cell, then rerun this section.

In [ ]:
SELECTED_PATHWAY = "ICAM"  # set to None to use the top-ranked pathway
DRIVER_TOP_N_LR = 15
DRIVER_TOP_N_EDGES = 35
DRIVER_EDGE_P_MAX = 0.05  # set to None to rank without an edge-level p filter

if analysis_tables is not None:
    pathway_name = SELECTED_PATHWAY or selected_pathway
    driver_figure, driver_axes, pathway_lr_review, pathway_edge_review = (
        plot_pathway_driver_summary(
            analysis_tables["pathway_edge_drivers"],
            contrast=selected_contrast,
            pathway=pathway_name,
            top_n_lr=DRIVER_TOP_N_LR,
            top_n_edges=DRIVER_TOP_N_EDGES,
            edge_exact_p_max=DRIVER_EDGE_P_MAX,
        )
    )
    display(driver_figure)
    plt.close(driver_figure)

    lr_columns = [
        "lr_label", "absolute_effect_percent", "signed_effect_sum", "direction_balance"
    ]
    edge_columns = [
        "source", "target", "ligand", "receptor", "difference_b_minus_a",
        "exact_permutation_p", "exact_permutation_q", "presence_class",
    ]
    display(pathway_lr_review[lr_columns])
    display(pathway_edge_review[edge_columns])
    print("Selected pathway:", selected_pathway)
else:
    print("Analysis tables are not present yet.")

## Directional chord plot

The chord diagram summarizes the selected pathway's directed cell–cell signaling edges. It uses the same pathway and contrast controls as the driver and bubble plots.

Rerun the rendering cell after changing the pathway; the title and output file are generated from the active selection.

In [ ]:
RENDER_RESULT_CHORD = True
CHORD_TOP_N = 15
if RENDER_RESULT_CHORD:
    if analysis_tables is None:
        raise RuntimeError("Run or load the pairwise analysis before rendering a chord")
    chord_input = prepare_chord_input(
        analysis_tables["pathway_edge_drivers"],
        contrast=selected_contrast,
        pathway_name=selected_pathway,
        top_n=CHORD_TOP_N,
    )
    chord_dir = ANALYSIS_DIR / "figures/chords"
    rendered = render_chord(
        chord_input,
        script_path=CHORD_SCRIPT,
        input_csv=chord_dir / f"{selected_contrast}__{selected_pathway}.csv",
        output_path=chord_dir / f"{selected_contrast}__{selected_pathway}.png",
        title=f"{selected_pathway}: {selected_contrast}",
        rscript_bin=config.get("runtime", {}).get("rscript_bin", "Rscript"),
    )
    display(Image(filename=str(rendered), width=900))
else:
    example_plot = REPO_ROOT / "docs/figures/liana_window_rankagg_example_chord.png"
    if example_plot.exists():
        display(Image(filename=str(example_plot), width=900))
    else:
        print("Example chord has not been rendered yet:", example_plot)

## Interpretation notes

- Biological samples, not spatial windows or cells, are the statistical replicates.
- Overlapping windows change spatial weighting but do not increase sample size.
- A completed inactive edge represents an interaction not emitted under the configured gates, not proof of literal biological absence.
- Pathway and edge results should be interpreted with their support counts and attainable exact-p resolution.